## Initiate packages

In [1]:
import openai
import groq
import os
from abc import ABC, abstractmethod
from typing import Dict, Any, Optional, Protocol
import time
import json
import re

## Insert Keys

In [2]:
OPENAI_API_KEY = "" ##key in your OpenAI API Key (https://platform.openai.com/api-keys)
GROQ_API_KEY = "" ##key in your Groq API Key (https://console.groq.com/keys)

## LLM Adapters
Only OpenAI and Groq for now, can add others as you desire

In [3]:
class LLMClientAdapter(Protocol):
    def generate(self, prompt: str, max_tokens: int = 256, **kwargs) -> Dict[str, Any]:
        """All adapters must implement this"""
        ...

In [4]:
class OpenAIAdapter(LLMClientAdapter):
    def __init__(self, api_key: str, model: str = "gpt-5-mini"):
        self.client = openai.OpenAI(api_key=api_key)
        self.model = model

    def generate(self, prompt: str, max_tokens: int = 256, **kwargs) -> Dict[str, Any]:
        resp = self.client.responses.create(
          model=model,
          input=prompt,
          max_tokens=max_tokens
        )
        text = resp.choices[0].message.content
        return {"text": text}

In [5]:
class GroqAdapter(LLMClientAdapter):
    def __init__(self, api_key: str, model: str = "meta-llama/llama-4-maverick-17b-128e-instruct"):
        self.client = groq.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt: str, max_tokens: int = 256, **kwargs) -> Dict[str, Any]:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            **kwargs
        )
        text = resp.choices[0].message.content
        return {"text": text}

In [6]:
groq_adapter = GroqAdapter(api_key=GROQ_API_KEY)
openai_adapter = OpenAIAdapter(api_key=OPENAI_API_KEY)

## Define Agents

In [7]:
class PlannerAgent:
    def __init__(self, client_adapter: LLMClientAdapter, role: str = "planning_agent"):
        self.client_adapter = client_adapter
        self.role = role

    def forward(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """
        Context given:
        1. Query

        Output:
        {
         "plan" : plan,
         }
        """
        print(f"running {self.role}")
        query = payload.get("query", "")

        # build prompt (or messages) for the client
        prompt = f"""Using the given query, determine the steps required to achieve the goal of the query:
        You must list out each step in bullet hyphen format (using "-").

        Query:
        {query}
        """

        start = time.time()
        try:
            resp = self.client_adapter.generate(prompt, max_tokens=1024)
            print(f"\noutput from {self.role} response --> {resp}\n")
            plan = resp.get("text", "")

            # plan = self._parse_plan(text)
            duration_ms = int((time.time() - start) * 1000)
            print(f"completed {self.role}")
            return {
                "status": "ok",
                "data": {"plan": plan},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

        except Exception as e:
            print(e)
            duration_ms = int((time.time() - start) * 1000)
            return {
                "status": "error",
                "data": {"error": str(e)},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

In [8]:
class OrchestratorAgent():
    def __init__(self, client_adapter: LLMClientAdapter, role: str = "orchestrator_agent"):
        self.client_adapter = client_adapter
        self.role = role

    def forward(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        print(f"running {self.role}")
        """
        Context given:
        1. Query 
        2. Plan from planner agent
        3. Tools retrieved
        4. Tools available
        5. Tools output
        6. previous invalidated reason (if given)

        Retrieved context from tools:
        ---------------------
        <TOOL 1>
        ...
        ---------------------
        
        ---------------------
        <TOOL 2>
        ...
        ---------------------

        

        Output:
        {
         "action" : tool_selection/proceed,
         "data" : { "tools_required" : [...], / "output" : str }
         }
        """
        query = payload.get('query')
        plan = payload.get('plan')
        retrieved_tools = payload.get('retrieved_tools')
        available_tools = payload.get('available_tools')
        invalidated_reason = payload.get('invalidated_reason', "")
        tool_outputs = payload.get('tool_output')

        context = f"""query: {query}
        plan: {plan}
        available tools: {available_tools}"""

        if retrieved_tools:
            context += f"\nretrieved tools: {retrieved_tools}"
        if invalidated_reason:
            context += f"\ninvalidated reason: {invalidated_reason}"
        if tool_outputs:
            context += f"\nTool outputs: {tool_outputs}"
        
        prompt = f"""Based on the given context, 
        ----------------------
        CONTEXT
        ----------------------
        {context}

        ----------------------
        OUTPUT FROM TOOLS
        ----------------------
        {tool_outputs}

        Output the next action, either requiring more data / context via retrieving from different tools (tool selection) or proceeding to output if enough context and information is already given for the outpu
        
        ----------------------
        OUTPUT FORMAT / SCHEMA
        ----------------------
        {{
            "action" : tool_selection OR proceed,
            "data" : {{ "tools_required" : [... from available tools] (use when tool_selection), "output": ... (use when proceed)}} (this can be empty if action = proceed)
        }}

        You MUST follow the output format schema, and it has to have 'action' and 'data' as the only key in the dictionary json output.
        ONLY output the json
        """

        start = time.time()
        try:
            resp = parse_llm_resp(self.client_adapter.generate(prompt, max_tokens=1024).get('text'))
            print(f"\noutput from {self.role} response --> {resp}\n")
            action = resp.get("action", "")
            data = resp.get("data", "")

            # plan = self._parse_plan(text)
            duration_ms = int((time.time() - start) * 1000)
            print(f"completed {self.role}")
            return {
                "status": "ok",
                "data": {"action": action, "data": data},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

        except Exception as e:
            print(e)
            duration_ms = int((time.time() - start) * 1000)
            return {
                "status": "error",
                "data": {"error": str(e)},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

In [ ]:
class ToolSelectorAgent():
    def __init__(self, client_adapter: LLMClientAdapter, role: str = "tool_selector_agent"):
        self.client_adapter = client_adapter
        self.role = role

    def forward(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        print(f"running {self.role}")
        """
        Context given:
        1. Tools required
        2. Tools available

        Output:
        {
         "tools" : [...tools...],
         "tools_unavailable" : [...tools...]
         }
        """
        tools_required = payload.get("tools_required")
        available_tools = payload.get("available_tools")

        context = f"""required tools: {tools_required}
        available tools: {available_tools}"""
        
        prompt = f"""Based on the given context, 
        ----------------------
        CONTEXT
        ----------------------
        {context}

        Output the tools required for usage, and also the different query/data to be sent into the different tools.

        ----------------------
        OUTPUT FORMAT / SCHEMA
        ----------------------
        {{
            "tools": [{{"rag": {{"query": "..."}}, "memory": {{"query": "..."}}]
        }}

        You MUST follow the output format schema, and it has to have 'tools' as the only key in the dictionary json output.
        ONLY output the json
        """

        start = time.time()
        try:
            resp = parse_llm_resp(self.client_adapter.generate(prompt, max_tokens=1024).get('text'))
            print(f"\noutput from {self.role} response --> {resp}\n")
            tools = resp.get("tools", "")

            # plan = self._parse_plan(text)
            duration_ms = int((time.time() - start) * 1000)
            print(f"completed {self.role}")
            return {
                "status": "ok",
                "data": {"tools" : tools},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

        except Exception as e:
            print(e)
            duration_ms = int((time.time() - start) * 1000)
            return {
                "status": "error",
                "data": {"error": str(e)},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

#### Execution Agents are agents that can be called via tools. They require 'usage' so the Orchestrator and Tool Selector agents know when to call those agents

In [9]:
class ExecutionAgent(ABC):
    @abstractmethod
    def usage(self):
        """Describe when or how to use this agent."""
        pass
        
    @abstractmethod
    def forward(self):
        ...
        

In [11]:
class MemoryAgent(ExecutionAgent):
    """
    Simple memory implementation for reproducible purposes
    In-process memory store (no persistence).
    Public API via forward(payload):
      payload = {"op": "store"|"retrieve", "key":..., "value":..., "query":..., ...}
    """

    def __init__(self, role: str = "memory_agent"):
        self.store: Dict[str, Any] = {}
        self.role = role

    def forward(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """
        Context given:
        1. operation
        2. key

        Retrieved context from tools:
        ---------------------
        <TOOL 1>
        ...
        ---------------------
        
        ---------------------
        <TOOL 2>
        ...
        ---------------------

        

        Output:
        {
         "data" : data_output_from_memory
         }
        """
        print(f"running {self.role}")
        start = time.time()
        op = payload.get("op")
        key = payload.get("key")
        try:
            if op == "store":
                self._store(key, payload.get("value"))
                print(f"completed {self.role}")
                return {"status": "ok", "data": {"stored": True}, "meta": {"agent_id": self.role, "duration_ms": int((time.time()-start)*1000)}}
            elif op == "retrieve":
                val = self._retrieve(key)
                print(f"completed {self.role}")
                return {"status": "ok", "data": {"value": val}, "meta": {"agent_id": self.role, "duration_ms": int((time.time()-start)*1000)}}
            else:
                return {"status": "error", "data": {"error": "unknown_op"}, "meta": {"agent_id": self.role}}
        except Exception as e:
            return {"status": "error", "data": {"error": str(e)}, "meta": {"agent_id": self.role}}

    def _store(self, key: Optional[str], value: Any):
        if key is None:
            key = f"item:{len(self.store)+1}"
        self.store[str(key)] = value

    def _retrieve(self, key: Optional[str]):
        if key is None:
            return None
        return self.store.get(str(key))

    def usage(self):
        return "Use this tool when the query requires recalling user-specific information, past conversations, or maintaining continuity across interactions."


In [12]:
from typing import List, Dict, Any, Optional
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

class InMemoryVectorStore:
    """
    Fast demo vector store using TF-IDF + cosine similarity.
    Methods:
      - add_documents(docs: List[Dict]) where each doc has {'id','text'}
      - search(query, k=5) -> list of documents with score
      - persist(path) / load(path)
    """

    def __init__(self):
        self.docs: List[Dict[str, Any]] = []
        self._texts: List[str] = []
        self._ids: List[str] = []
        self.vectorizer: Optional[TfidfVectorizer] = None
        self._emb_matrix = None

    def add_documents(self, docs: List[Dict[str, Any]]):
        """
        docs: list of {'id': str, 'text': str, ...}
        """
        for d in docs:
            self.docs.append(d)
            self._texts.append(str(d.get("text", "")))
            self._ids.append(d.get("id"))
        # rebuild vectorizer (fast enough for demo)
        self.vectorizer = TfidfVectorizer().fit(self._texts)
        self._emb_matrix = self.vectorizer.transform(self._texts)

    def search(self, query: str, k: int = 5) -> List[Dict[str, Any]]:
        if not self.docs or self.vectorizer is None:
            return []
        q_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(q_vec, self._emb_matrix).flatten()
        idxs = np.argsort(-sims)[:k]
        results = []
        for i in idxs:
            results.append({"id": self._ids[i], "text": self._texts[i], "score": float(sims[i]), "doc": self.docs[i]})
        return results

In [13]:
class RAGAgent(ExecutionAgent):
    """
    Simple retrieval agent wrapper over an InMemoryVectorStore.
    Exposes forward(payload) where payload = {"query":..., "k": int}
    """

    def __init__(self, vector_store, role: str = "retrieval_agent"):
        self.store = vector_store
        self.role = role

    def forward(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """
        Context given:
        1. Query
        2. Top k

        Output:
        {
            "documents" : top_k_doc_outputs
         }
        """
        print(f"running {self.role}")
        start = time.time()
        q = payload.get("query", "")
        k = int(payload.get("k", 5))
        try:
            results = self.store.search(q, k=k)
            print(f"\noutput from {self.role} response --> {results}\n")
            duration_ms = int((time.time() - start)*1000)
            docs = []
            for r in results:
                doc = r.get("doc", {})
                doc_out = {
                    "id": doc.get("id"),
                    "content": doc.get("text") or doc.get("content"),
                    "score": r.get("score"),
                    "snippet": (doc.get("text") or "")[:300]
                }
                docs.append(doc_out)
            print(f"completed {self.role}")
            return {"status": "ok", "data": {"documents": docs}, "meta": {"agent_id": self.role, "duration_ms": duration_ms}}
        except Exception as e:
            return {"status": "error", "data": {"error": str(e)}, "meta": {"agent_id": self.role}}

    def usage(self):
        return "Use this tool when the query requires retrieving knowledge from external documents, databases, or APIs to provide grounded, up-to-date, or domain-specific information."

In [14]:
class ValidationAgent():
    def __init__(self, client_adapter: LLMClientAdapter, role: str = "validation_agent"):
        self.client_adapter = client_adapter
        self.role = role

    def forward(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """
        Context given:
        1. Query 
        2. Output

        Output:
        {
         "validated" : "yes/no",
         "invalidated_reason" : str
         }
        """
        print(f"running {self.role}")
        query = payload.get('query')
        output = payload.get('output')

        context = f"""query: {query}
        output: {output}"""
        
        prompt = f"""Based on the given context, 
        ----------------------
        CONTEXT
        ----------------------
        {context}


        Output whether or not the given output is validated (this is determined by whether the output answers the query given). If invalidated, give the invalidated reason in the output.

        ----------------------
        OUTPUT FORMAT / SCHEMA
        ----------------------
        {{
            "validated" : "yes/no",
            "invalidated_reason" : "..." if invalidated
        }}

        You MUST follow the output format schema, and it has to have 'validated' and 'invalidated_reason' as the only key in the dictionary json output.
        ONLY output the json
        """

        start = time.time()
        try:
            resp = parse_llm_resp(self.client_adapter.generate(prompt, max_tokens=1024).get('text'))
            print(f"\noutput from {self.role} response --> {resp}\n")
            validated = resp.get("validated", "")
            invalidated_reason = resp.get("invalidated_reason", "")

            # plan = self._parse_plan(text)
            duration_ms = int((time.time() - start) * 1000)
            print(f"completed {self.role}")
            return {
                "status": "ok",
                "data": {"validated": validated, "invalidated_reason": invalidated_reason},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

        except Exception as e:
            duration_ms = int((time.time() - start) * 1000)
            print(e)
            return {
                "status": "error",
                "data": {"error": str(e)},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

In [15]:
class OutputFormatAgent():
    def __init__(self, client_adapter: LLMClientAdapter, role: str = "format_agent"):
        self.client_adapter = client_adapter
        self.role = role

    def forward(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """
        Context given:
        1. Query
        2. Output

        Given the output, convert it to markdown format with the same response.

        Output:
        string format - response in markdown format
        """
        print(f"running {self.role}")
        query = payload.get('query')
        output = payload.get('output')

        context = f"""query: {query}
        output: {output}"""
        
        prompt = f"""Based on the given context, 
        ----------------------
        CONTEXT
        ----------------------
        {context}


        Given the output, convert it to markdown format with the same response.

        ----------------------
        OUTPUT FORMAT / SCHEMA
        ----------------------
        string format in markdown

        ONLY output the string output
        """

        start = time.time()
        try:
            resp = self.client_adapter.generate(prompt, max_tokens=1024).get('text')
            print(f"\noutput from {self.role} response --> {resp}\n")
            text_output = resp

            # plan = self._parse_plan(text)
            duration_ms = int((time.time() - start) * 1000)
            print(f"completed {self.role}")
            return {
                "status": "ok",
                "data": {"output": text_output},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

        except Exception as e:
            print(e)
            duration_ms = int((time.time() - start) * 1000)
            return {
                "status": "error",
                "data": {"error": str(e)},
                "meta": {"agent_id": self.role, "duration_ms": duration_ms}
            }

## Create a registry for these agents for easy handling

In [16]:
class AgentRegistry:
    def __init__(self, client_adapter, vector_store):
        self.client_adapter = client_adapter
        self.store = vector_store
        self._registry = {
            "planning_agent": lambda: PlannerAgent(self.client_adapter),
            "orchestrator_agent": lambda: OrchestratorAgent(self.client_adapter),
            "validation_agent": lambda: ValidationAgent(self.client_adapter),
            "rag_agent": lambda: RAGAgent(self.store),
            "format_agent": lambda: OutputFormatAgent(self.client_adapter),
            "memory_agent": lambda: MemoryAgent(),
            "tool_selector_agent": lambda: ToolSelectorAgent(self.client_adapter)
        }

    def get(self, role: str):
        if role not in self._registry:
            raise ValueError(f"No agent registered for role '{role}'")
        return self._registry[role]()

In [17]:
vs = InMemoryVectorStore()
docs = [
    {"id": "d1", "text": "Our refund policy: refunds within 30 days with receipt."},
    {"id": "d2", "text": "Returns require the original packaging and a receipt."},
    {"id": "d3", "text": "For international orders, returns take up to 60 days."},
]
vs.add_documents(docs)

In [18]:
registry = AgentRegistry(client_adapter = groq_adapter, vector_store = vs)

In [19]:
planning_agent = registry._registry['planning_agent']()
orchestrator_agent = registry._registry['orchestrator_agent']()
validation_agent = registry._registry['validation_agent']()
rag_agent = registry._registry['rag_agent']()
format_agent = registry._registry['format_agent']()
memory_agent = registry._registry['memory_agent']()
tool_selector_agent = registry._registry['tool_selector_agent']()

## Run multi-agent flow

In [24]:
def run_orchestration(orchestrator_agent, tool_selector_agent, query, plan, available_tools, invalidated_reason, retrieved_tools):
    tools_output = {"tools": []}
    payload = {
            "query": query, 
            "available_tools" : available_tools,
            "plan" : plan
        }
    if invalidated_reason:
        payload["invalidated_reason"] = invalidated_reason
    if tools_output.get("tools"):
        payload["tools_output"] = tools_output
    if retrieved_tools:
        payload["retrieved_tools"] = retrieved_tools
    while True:
        orch_output = orchestrator_agent.forward(payload).get("data")
        action = orch_output.get("action")
        if action == "proceed":
            return orch_output.get("data").get("output")
        tools_required = orch_output.get("tools_required")
        tools_to_use = tool_selector_agent.forward({
            "tools_required" : tools_required,
            "query" : query
        }).get("data").get("tools")
    
    for agent_name, query in tools_to_use:
        tool_agent = registry._registry.get(agent_name) if hasattr(registry, "_registry") else registry.get(agent_name)
        if tool_agent is None:
            # handle missing agent
            results["tools"].append((agent_name, {"error": f"agent '{agent_name}' not found"}))
            continue
        try:
            output = tool_agent.forward(query)
        except Exception as e:
            tools_output["tools"].append((agent_name, {"error": str(e)}))
        else:
            tools_output["tools"].append((agent_name, output))

In [32]:
def parse_llm_resp(text):
    print(f"\n PARSING... {text}\n")
    cleaned = re.sub(r"^```json\s*|\s*```$", "", text.strip(), flags=re.DOTALL)
    return json.loads(cleaned)

In [33]:
query = "hello"

In [34]:
invalidated_reason = False
plan = planning_agent.forward({"query": query}).get("data").get("plan")
available_tools = [f"{role} - {factory().usage()}" for role, factory in registry._registry.items() if issubclass(factory().__class__, ExecutionAgent)]
retrieved_tools = []
while True:
    output = run_orchestration(orchestrator_agent, tool_selector_agent, query, plan, available_tools, invalidated_reason, retrieved_tools)
    validated, invalidated_reason = validation_agent.forward({"query" : query, "output": output}).get("data")
    if validated:
        break
formatted_output = format_agent.forward({"query" : query, "output": output})

running planning_agent

output from planning_agent response --> {'text': '- Understand the query: The given query is a greeting, "hello".\n- Analyze the query for a specific task or question: The query "hello" does not contain a specific task or question.\n- Determine the context or intent behind the query: The intent appears to be a greeting rather than a request for information or action.\n- Since the query is a greeting and not a specific request, identify a suitable response: A response to a greeting is typically another greeting or acknowledgement.\n- List out steps to achieve the goal (in this case, responding appropriately to the greeting):\n- Recognize that "hello" is a greeting.\n- Respond with a greeting or acknowledgement.\n \nSince the task is to list steps in bullet hyphen format, here\'s the response in the required format:\n- Respond to the greeting "hello".\n- Acknowledge the greeting.\n- Provide a suitable greeting in response, such as "hello" or a similar greeting. \n

In [38]:
final_output = formatted_output['data']['output']

In [39]:
print(final_output)

## Hello! How can I assist you today?
